In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
base_dir = "/path/to/project/"
# the file has a column name already
inversion = os.path.join(base_dir, "INV_len1kb_0.1-0.9_CHA.tsv")
df = pd.read_csv(inversion, sep="\t", header=0)
df

In [ ]:
df = df[~df["CHR"].isin(["chrX", "chrY", "chrM"])]
df['chr_n'] = df["CHR"].str.replace("chr", "", regex=True)

In [ ]:
df = df[["TOOL", "CHR", "START", "END", "LENGTH", "AF_CHA", "chr_n"]].copy()
df

In [ ]:
def weighted_mean_rate(map_df, start, end):
    """Weighted mean Rec.Rate over [start, end) using overlap length as weights."""
    ov = map_df[(map_df["End"] > start) & (map_df["Start"] < end)].copy()
    if ov.empty:
        return np.nan
    ov["Start"] = np.maximum(ov["Start"].to_numpy(), start)
    ov["End"]   = np.minimum(ov["End"].to_numpy(),   end)
    w = (ov["End"] - ov["Start"]).to_numpy()
    # safety: if all weights are 0, return NaN
    if w.sum() == 0:
        return np.nan
    return np.average(ov["Rec.Rate"].to_numpy(), weights=w)



In [ ]:
base_dir_cha = "/path/to/project/"
cha_pan = os.path.join(base_dir_cha, "final_analysis/data/CHA/pan/bp35w60")

chroms = [str(i) for i in range(1,23)]

maps_pan = {}
chrom_sizes = {}

for chrom in chroms:
    p = os.path.join(cha_pan, f"CHA_recombmap_chr{chrom}_bp35w60")
    dfm = pd.read_csv(p, sep="\t", header=None, names=["Start","End","Rec.Rate"])
    maps_pan[chrom] = dfm
    chrom_sizes[chrom] = int(dfm["End"].max())

print("Loaded CHA recomb maps")

In [ ]:
pan_available = "CHM13v2.telo_cent.complement.bed"

pan_available_region = pd.read_csv(
	pan_available, sep="\t", header=None, names=["Chrom", "Start", "End"]
)
# remove chrX and chrY in Chrom
pan_available_region = pan_available_region[~pan_available_region["Chrom"].isin(["chrX", "chrY"])]
pan_available_region ["chr"] = pan_available_region ["Chrom"].str.replace("chr", "")
pan_available_region

In [ ]:
def clip_map_to_allowed_regions(df_map: pd.DataFrame,
                               allowed_df: pd.DataFrame,
                               chrom: str,
                               chrom_col_allowed: str = "chr",
                               start_col_allowed: str = "Start",
                               end_col_allowed: str = "End") -> pd.DataFrame:

    # Allowed intervals for this chromosome
    allowed = allowed_df[allowed_df[chrom_col_allowed].astype(str) == str(chrom)][
        [start_col_allowed, end_col_allowed]
    ].copy()

    if allowed.empty or df_map.empty:
        return df_map.iloc[0:0].copy()

    # sort for safety
    allowed = allowed.sort_values([start_col_allowed, end_col_allowed]).to_numpy()
    df_map = df_map.sort_values(["Start", "End"]).reset_index(drop=True)

    out_rows = []

    # Two-pointer sweep (fast enough; map windows are usually not huge)
    j = 0
    for s, e, r in df_map[["Start", "End", "Rec.Rate"]].to_numpy():
        if e <= s:
            continue

        # advance allowed pointer until it might overlap
        while j < len(allowed) and allowed[j][1] <= s:
            j += 1

        k = j
        # collect all overlaps with allowed intervals
        while k < len(allowed) and allowed[k][0] < e:
            a_s, a_e = allowed[k]
            ov_s = max(s, a_s)
            ov_e = min(e, a_e)
            if ov_s < ov_e:
                out_rows.append((ov_s, ov_e, r))
            if a_e >= e:
                break
            k += 1

    if not out_rows:
        return df_map.iloc[0:0].copy()

    df_out = pd.DataFrame(out_rows, columns=["Start", "End", "Rec.Rate"])
    df_out = df_out.sort_values(["Start", "End"]).reset_index(drop=True)
    return df_out

In [ ]:
for chrom in chroms:
	maps_pan[str(chrom)] = clip_map_to_allowed_regions(
        df_map=maps_pan[str(chrom)],
        allowed_df=pan_available_region,
        chrom=str(chrom),          
        chrom_col_allowed="chr",   
        start_col_allowed="Start",
        end_col_allowed="End"
    )

In [ ]:
for df_inv in [df]:
	rec_rates = []
	for idx, row in df_inv.iterrows():
		chrom = row["chr_n"]
		start = row["START"]
		end = row["END"]
		map_df = maps_pan[chrom]
		wmrr = weighted_mean_rate(map_df, start, end)
		rec_rates.append(wmrr)
	df_inv["mean_rec_rate"] = rec_rates

In [ ]:
df

In [ ]:
import random

for df_inv in [df]:
	random_rec_rates = []
	for idx, row in df_inv.iterrows():
		chrom = row["chr_n"]
		length = row["LENGTH"]
		chrom_size = chrom_sizes[chrom]
		# randomly sample a start position
		max_start = chrom_size - length
		rand_start = random.randint(0, max_start)
		rand_end = rand_start + length
		map_df = maps_pan[chrom]
		wmrr = weighted_mean_rate(map_df, rand_start, rand_end)
		# if wmrr is na, resample until not na
		while pd.isna(wmrr):
			rand_start = random.randint(0, max_start)
			rand_end = rand_start + length
			wmrr = weighted_mean_rate(map_df, rand_start, rand_end)
		random_rec_rates.append(wmrr)
	df_inv["random_mean_rec_rate"] = random_rec_rates

In [ ]:
df

In [ ]:
from scipy.stats import wilcoxon
import numpy as np

inv = df["mean_rec_rate"].to_numpy()
rnd = df["random_mean_rec_rate"].to_numpy()

# remove pairs with identical values (Wilcoxon requirement)
mask = (inv != rnd) & (~np.isnan(inv)) & (~np.isnan(rnd))

stat, p = wilcoxon(
    inv[mask],
    rnd[mask],
    alternative="less",  # inv < random → coldspot
    zero_method="wilcox"
)

stat, p

In [ ]:
def chrom_mean_rate(map_df):
    window_lens = map_df["End"] - map_df["Start"]
    return (window_lens * map_df["Rec.Rate"]).sum() / window_lens.sum()

chrom_mean_rates = {}

for chrom, map_df in maps_pan.items():
    chrom_mean_rates[chrom] = chrom_mean_rate(map_df)

In [ ]:
df["chrom_mean_rec_rate"] = (
    df["chr_n"].map(chrom_mean_rates)
)
df

In [ ]:
from scipy.stats import binomtest

df["inv_colder"] = (
    df["mean_rec_rate"] < df["random_mean_rec_rate"]
)

k = df["inv_colder"].sum()
print(k)

res = binomtest(k, len(df), p=0.5, alternative="greater")
print(res)

In [ ]:
# write a csv output of the table
output_path = os.path.join(base_dir, "inversion_rec_rate_comparison.csv")
df.to_csv(output_path, index=False)